# Optimization States and Reconstructed Solutions


```{contents}
:local:
:depth: 2
```


```{index} Define the State from the Choices That Remain
```



```{index} Derive Every Entry of a Small Purchasing Plan
```

## Derive Every Entry of a Small Purchasing Plan


A purchaser can use unlimited packs containing one, three or four items and needs exactly six items. Largest-first gives 4+1+1, but 3+3 uses fewer packs. Dynamic programming improves the choice by comparing every legal final pack, not by repeatedly taking the largest.

Let best[a] be the minimum number of packs totaling a. Best[0]=0 represents the valid empty purchase. For each positive a, consider a final pack c with c≤a. Only reachable best[a-c] entries can supply candidates; their count plus one is a possible answer. Store both the lowest count and the chosen last pack. Null means unreachable, not zero packs.

| Amount a | Reachable candidates using last pack 1, 3, 4 | best[a] | Chosen last pack |
| --- | --- | ---: | ---: |
| 0 | Base case | 0 | None |
| 1 | best[0]+1=1 | 1 | 1 |
| 2 | best[1]+1=2 | 2 | 1 |
| 3 | best[2]+1=3; best[0]+1=1 | 1 | 3 |
| 4 | best[3]+1=2; best[1]+1=2; best[0]+1=1 | 1 | 4 |
| 5 | best[4]+1=2; best[2]+1=3; best[1]+1=2 | 2 | 1 |
| 6 | best[5]+1=3; best[3]+1=2; best[2]+1=3 | 2 | 3 |

The tie at amount five preserves last pack one when denominations are visited in ascending order and updates require a strict improvement. This is a deterministic witness policy; it does not change the optimum count. The following focused code uses a fixed valid input to expose the table updates. The retained full API performs validation and resource-bound checks.


In [1]:
using System;
using System.Collections.Generic;
int[] packs = { 1, 3, 4 };
int target = 6;
int?[] best = new int?[target + 1];
int[] last = new int[target + 1];
best[0] = 0;
for (int amount = 1; amount <= target; amount++)
{
    foreach (int pack in packs)
    {
        if (pack > amount || best[amount - pack] is not int previous) continue;
        int candidate = previous + 1;
        if (!best[amount].HasValue || candidate < best[amount].GetValueOrDefault())
        {
            best[amount] = candidate;
            last[amount] = pack;
        }
    }
    Console.WriteLine($"Amount {amount}: count {best[amount]}, last {last[amount]}");
}
var plan = new List<int>();
if (best[target].HasValue)
    for (int remaining = target; remaining > 0; remaining -= last[remaining])
        plan.Add(last[remaining]);
Console.WriteLine($"Plan: {string.Join("+", plan)}");


Amount 1: count 1, last 1
Amount 2: count 2, last 1
Amount 3: count 1, last 3
Amount 4: count 1, last 4
Amount 5: count 2, last 1
Amount 6: count 2, last 3
Plan: 3+3


Read the recurrence in the code: amount-pack selects the smaller state; previous+1 accounts for the final pack; the comparison retains the best candidate. A positive pack makes the dependency strictly smaller, so increasing amounts are a valid evaluation order. Zero or negative denominations would invalidate this reasoning and are rejected by the full API.

Reconstruction follows decisions backwards, not all table entries in descending order. For six, last[6]=3, so append three and continue with remaining three. Last[3]=3 reaches remaining zero. The two chosen packs sum to six and their count equals best[6]. A valid plan proves feasibility; the recurrence's comparison of all final choices establishes optimality.


```{index} Guide an Unreachable Entry and Check Its Witness
```

## Guide an Unreachable Entry and Check Its Witness


Remove denomination one and ask for five using only three and four. Fill best[0] through best[5] before running a program. Do not initialize all counts to zero: that would make impossible remainders appear reachable.

```{dropdown} Check the smaller table
The counts are 0, null, null, 1, 1, null. Five is unreachable because neither remainder two nor remainder one is reachable. No reconstruction should start from five. Target zero is different: its count is zero and its valid plan is empty.
```

Now compare two records each worth three with unlimited denomination three. A target of nine is reachable with unlimited packs, but not from two one-use records. Amount alone is sufficient only when the available choices are unchanged after each use. The next backtracking lesson includes the position/inventory information needed for one-use choices.

Before the independent purchasing assignments, validate a candidate witness yourself: every pack is allowed, all values are positive, their total is the target, and their count matches the reported optimum. Then use a small independent reference to check the optimum. A program that prints two without a matching feasible witness has not completed the result contract.


```{index} Define the State from the Choices That Remain
```

## Define the State from the Choices That Remain



A purchasing team can buy unlimited packs of sizes one, three and four and wants exactly six units using the fewest packs. Greedily taking four first leaves two one-unit packs, for three packs total. Two packs of three use only two packs. The counterexample shows that a locally largest pack is not a generally safe choice for this objective.

Let B(a) be the minimum number of packs for amount a. The base is B(0)=0. To solve a positive amount, consider every permitted last pack c and a reachable predecessor amount a-c. A candidate solution has one more pack than B(a-c). Choose the minimum candidate. This considers the last step of every feasible optimum, rather than guessing which first pack looks largest.

| Amount | Best count for sizes 1,3,4 | One predecessor choice |
| --- | --- | --- |
| 0 | 0 | Empty solution |
| 1 | 1 | 0 plus 1 |
| 2 | 2 | 1 plus 1 |
| 3 | 1 | 0 plus 3 |
| 4 | 1 | 0 plus 4 |
| 5 | 2 | 1 plus 4, or another optimal tie |
| 6 | 2 | 3 plus 3 |

Optimal substructure supplies the justification: removing the last pack from an optimal solution leaves an optimal solution to the predecessor amount. If a better predecessor existed, appending the same last pack would improve the supposed optimum. Positive pack sizes make predecessor amounts smaller, giving a valid tabulation order and terminating reconstruction.

For related explanation, see the further reading below.[^development-reading]


```{index} Preserve Enough Information to Explain the Answer
```



```{index} Preserve Enough Information to Explain the Answer
```

## Preserve Enough Information to Explain the Answer



An optimum count is not the same output as an actual purchasing plan. Store a predecessor choice when an amount is improved, then walk backward to zero to reconstruct a witness. Validate that the witness sums to the requested amount and has the reported number of packs. If no predecessor is feasible, represent unreachable distinctly from the valid zero-pack solution at amount zero.

The state depends on inventory assumptions. If each invoice or pack position can be used only once, amount alone does not identify the remaining options. Chapter 25's backtracking lesson uses both a position and remaining target. Reusing an amount-only unlimited-pack recurrence for limited inventory can return an impossible plan.

Guided variation: remove size one and request five from sizes three and four. Explain why zero is not a valid optimum count for this request and why reconstruction must not loop forever through an invented predecessor. Then request zero and distinguish that valid empty solution from the unreachable positive amount. The preserved verification cases check both objective value and witness.


```{index} Minimum Coins: Model Before Implementation
```

## Minimum Coins: Model Before Implementation




```{index} Minimum Coins: Model Before Implementation
```

A purchasing system uses unlimited packs of sizes 1,3,4. Minimize the number of packs totaling target six. Let B(a) be the minimum count for a; B(0)=0, and B(a)=min(1+B(a−c)) over positive denominations c≤a whose remainder is reachable. If no such predecessor exists, a is unreachable. Removing the last coin from any optimal plan leaves an optimal remainder plan: a better remainder would improve the original plan.

In [ ]:
using System;
using System.Linq;
using System.Collections.Generic;

var plan=CoinPlanning.MinimumCoins(new[]{1,3,4},6);
Console.WriteLine($"minimum={plan.Count}, coins={string.Join("+",plan.Coins)}");
Console.WriteLine("table="+string.Join(",",plan.Best.Select(x=>x?.ToString()??"unreachable")));
var impossible=CoinPlanning.MinimumCoins(new[]{4,6},7);
Console.WriteLine($"seven reachable={impossible.Count is not null}, witness length={impossible.Coins.Length}");
public static class CoinPlanning
{
    public record CoinPlan(int? Count,int[] Coins,int?[] Best);
    // Unlimited positive denominations. A target cap bounds table allocation.
    public static CoinPlan MinimumCoins(int[] denominations,int target)
    {
        ArgumentNullException.ThrowIfNull(denominations);
        if(target<0||target>1_000_000)throw new ArgumentOutOfRangeException(nameof(target));
        if(denominations.Any(c=>c<=0))throw new ArgumentException("Coins must be positive.");
        int[] coins=denominations.Distinct().OrderBy(c=>c).ToArray();
        var best=new int?[target+1];var last=new int[target+1];best[0]=0;
        for(int amount=1;amount<=target;amount++)
        {
            foreach(int coin in coins)
            {
                if(coin>amount)break;
                if(best[amount-coin] is not int count)continue;
                int candidate=count+1;
                if(best[amount] is null||candidate<best[amount])
                {best[amount]=candidate;last[amount]=coin;}
            }
        }
        var chosen=new List<int>();
        if(best[target] is not null)
            for(int remaining=target;remaining>0;remaining-=last[remaining])chosen.Add(last[remaining]);
        return new CoinPlan(best[target],chosen.ToArray(),best);
    }
}


Positive denominations ensure a−c<a, so amounts in increasing order satisfy dependencies and reconstruction strictly decreases remaining amount. Nullable count distinguishes unreachable from a valid zero-coin plan. Duplicate denominations are normalized; ascending denominations and strict improvement make equal-count ties deterministic. Empty denominations can make only zero. Reject zero/negative coins: a zero coin breaks progress and a negative coin changes the state graph. The table invariant is that all smaller amounts have correct optima or correct unreachable markers; considering every possible last coin establishes the next entry.

```{index} State Size, Numeric Size, and the Requested Result
```

## State Size, Numeric Size, and the Requested Result




```{index} State Size, Numeric Size, and the Requested Result
```

With target T and D input denominations, sorting/normalization costs O(D log(D+1)); filling the table costs O(Td) for d distinct denominations, and table storage is O(T). Reconstructing K chosen coins adds O(K) output space and time. This implementation caps T at one million as an explicit allocation policy. These costs depend on the numeric target, not just the number of digits encoding it: the method is pseudo-polynomial.

Memoization can avoid some states reachable from the requested state; tabulation can avoid recursion and simplify dependency order. Neither is universally better. An amount-only coin state is sufficient for unlimited coins. If each pack is available once, remaining inventory matters: amount alone loses essential information. Chapter 25’s subset search uses (position, remaining), which preserves which items are still available. Rolling storage also trades away reconstruction information unless another recovery method is supplied.

```{index} Exercise
```

## Exercise




```{index} Exercise
```

Check zero, unreachable, duplicate, and invalid denomination contracts. Explain why unlimited coins require a different state from one-use invoices.

In [ ]:
using System;
using System.Linq;
using System.Collections.Generic;

int[] denominations={3,3,4};
// Your code starts here.
/* TODO: complete the task. */
// Your code ends here.

In [1]:
using System;
using System.Linq;
using System.Collections.Generic;

// Solution
int[] denominations={3,3,4};
foreach(int target in new[]{0,2,6})
{
 var p=CoinPlanning.MinimumCoins(denominations,target);
 Console.WriteLine($"target={target}, count={p.Count?.ToString()??"unreachable"}, witness=[{string.Join(",",p.Coins)}]");
}
try{CoinPlanning.MinimumCoins(new[]{0,3},6);}catch(ArgumentException){Console.WriteLine("zero denomination rejected");}
public static class CoinPlanning
{
    public record CoinPlan(int? Count,int[] Coins,int?[] Best);
    // Unlimited positive denominations. A target cap bounds table allocation.
    public static CoinPlan MinimumCoins(int[] denominations,int target)
    {
        ArgumentNullException.ThrowIfNull(denominations);
        if(target<0||target>1_000_000)throw new ArgumentOutOfRangeException(nameof(target));
        if(denominations.Any(c=>c<=0))throw new ArgumentException("Coins must be positive.");
        int[] coins=denominations.Distinct().OrderBy(c=>c).ToArray();
        var best=new int?[target+1];var last=new int[target+1];best[0]=0;
        for(int amount=1;amount<=target;amount++)
        {
            foreach(int coin in coins)
            {
                if(coin>amount)break;
                if(best[amount-coin] is not int count)continue;
                int candidate=count+1;
                if(best[amount] is null||candidate<best[amount])
                {best[amount]=candidate;last[amount]=coin;}
            }
        }
        var chosen=new List<int>();
        if(best[target] is not null)
            for(int remaining=target;remaining>0;remaining-=last[remaining])chosen.Add(last[remaining]);
        return new CoinPlan(best[target],chosen.ToArray(),best);
    }
}


target=0, count=0, witness=[]
target=2, count=unreachable, witness=[]
target=6, count=2, witness=[3,3]
zero denomination rejected


```{index} Dynamic Programming and Backtracking Lab
```

<span id="dynamic-programming-and-backtracking-lab"></span>


Use a purchasing workflow to compare unlimited pack quantities with one-use invoice adjustments. Verify objective values and witnesses independently. Download the complete implementations and reference checks: [PlanningChecks.cs](../../demos/25/PlanningChecks.cs).

```{index} Verify an Optimum and Its Witness
```

## Verify an Optimum and Its Witness




```{index} Verify an Optimum and Its Witness
```

For pack sizes 1,3,4 and quantity six, largest-first makes 4+1+1. Dynamic programming returns a two-pack plan. Independently check allowed sizes, witness count, and total, rather than trusting only a printed minimum.

In [ ]:
using System;
using System.Linq;
using System.Collections.Generic;

int[] sizes={1,3,4};int target=6;
var plan=CoinPlanning.MinimumCoins(sizes,target);
if(plan.Count!=2||plan.Coins.Length!=plan.Count||plan.Coins.Any(c=>!sizes.Contains(c))||plan.Coins.Sum()!=target)throw new Exception("Invalid pack plan.");
Console.WriteLine($"minimum packs={plan.Count}, witness={string.Join("+",plan.Coins)}");
public static class CoinPlanning
{
    public record CoinPlan(int? Count,int[] Coins,int?[] Best);
    // Unlimited positive denominations. A target cap bounds table allocation.
    public static CoinPlan MinimumCoins(int[] denominations,int target)
    {
        ArgumentNullException.ThrowIfNull(denominations);
        if(target<0||target>1_000_000)throw new ArgumentOutOfRangeException(nameof(target));
        if(denominations.Any(c=>c<=0))throw new ArgumentException("Coins must be positive.");
        int[] coins=denominations.Distinct().OrderBy(c=>c).ToArray();
        var best=new int?[target+1];var last=new int[target+1];best[0]=0;
        for(int amount=1;amount<=target;amount++)
        {
            foreach(int coin in coins)
            {
                if(coin>amount)break;
                if(best[amount-coin] is not int count)continue;
                int candidate=count+1;
                if(best[amount] is null||candidate<best[amount])
                {best[amount]=candidate;last[amount]=coin;}
            }
        }
        var chosen=new List<int>();
        if(best[target] is not null)
            for(int remaining=target;remaining>0;remaining-=last[remaining])chosen.Add(last[remaining]);
        return new CoinPlan(best[target],chosen.ToArray(),best);
    }
}


A valid witness establishes an upper bound on the optimum; the recurrence proof or an independent exhaustive reference establishes that no better solution exists. Validation and optimality are different obligations.

```{index} Inventory Changes the Answer
```

## Inventory Changes the Answer




```{index} Inventory Changes the Answer
```

An unlimited pack can be reused. A one-use invoice can appear at most once. With two records both worth four, target twelve is feasible with unlimited denomination four but impossible using only those two records.

In [ ]:
using System;
using System.Linq;
using System.Collections.Generic;

int[] invoices={4,4};
Console.WriteLine($"unlimited count={CoinPlanning.MinimumCoins(invoices,12).Count}");
Console.WriteLine($"one-use feasible={SubsetPlanning.FindSubset(invoices,12).Indices is not null}");
var eight=SubsetPlanning.FindSubset(invoices,8);
Console.WriteLine($"eight IDs={string.Join(",",eight.Indices!)}, sum={eight.Indices!.Sum(i=>(long)invoices[i])}");
public static class CoinPlanning
{
    public record CoinPlan(int? Count,int[] Coins,int?[] Best);
    // Unlimited positive denominations. A target cap bounds table allocation.
    public static CoinPlan MinimumCoins(int[] denominations,int target)
    {
        ArgumentNullException.ThrowIfNull(denominations);
        if(target<0||target>1_000_000)throw new ArgumentOutOfRangeException(nameof(target));
        if(denominations.Any(c=>c<=0))throw new ArgumentException("Coins must be positive.");
        int[] coins=denominations.Distinct().OrderBy(c=>c).ToArray();
        var best=new int?[target+1];var last=new int[target+1];best[0]=0;
        for(int amount=1;amount<=target;amount++)
        {
            foreach(int coin in coins)
            {
                if(coin>amount)break;
                if(best[amount-coin] is not int count)continue;
                int candidate=count+1;
                if(best[amount] is null||candidate<best[amount])
                {best[amount]=candidate;last[amount]=coin;}
            }
        }
        var chosen=new List<int>();
        if(best[target] is not null)
            for(int remaining=target;remaining>0;remaining-=last[remaining])chosen.Add(last[remaining]);
        return new CoinPlan(best[target],chosen.ToArray(),best);
    }
}

public static class SubsetPlanning
{
    public record SubsetResult(int[]? Indices,int Calls,int Hits,int Prunes,int FailedStates);
    // Signed values are allowed; positions are distinct and each is used once.
    public static SubsetResult FindSubset(int[] values,long target,bool prune=true,bool memoize=true,int maxCalls=1_000_000)
    {
        ArgumentNullException.ThrowIfNull(values);
        if(values.Length>128)throw new ArgumentException("At most 128 positions in this teaching implementation.");
        if(maxCalls<1)throw new ArgumentOutOfRangeException(nameof(maxCalls));
        int[] data=(int[])values.Clone();int n=data.Length;
        var lower=new long[n+1];var upper=new long[n+1];
        for(int i=n-1;i>=0;i--)
        {lower[i]=lower[i+1]+Math.Min(0,data[i]);upper[i]=upper[i+1]+Math.Max(0,data[i]);}
        var failed=new HashSet<(int Index,long Remaining)>();var chosen=new List<int>();
        int calls=0,hits=0,prunes=0;int[]? answer=null;
        bool Search(int index,long remaining)
        {
            if(calls==maxCalls)throw new InvalidOperationException("Search budget exhausted; feasibility is unknown.");
            calls++;
            if(remaining==0){answer=chosen.ToArray();return true;}
            if(index==n)return false;
            if(prune&&(remaining<lower[index]||remaining>upper[index])){prunes++;return false;}
            var state=(index,remaining);
            if(memoize&&failed.Contains(state)){hits++;return false;}
            chosen.Add(index);bool found;
            try{found=Search(index+1,remaining-data[index]);}
            finally{chosen.RemoveAt(chosen.Count-1);}
            if(found||Search(index+1,remaining))return true;
            if(memoize)failed.Add(state);
            return false;
        }
        // Even an unpruned trace rejects targets outside the total signed range.
        if(target<lower[0]||target>upper[0]){calls=1;prunes=1;}
        else Search(0,target);
        if(chosen.Count!=0)throw new InvalidOperationException("Undo failed.");
        return new SubsetResult(answer,calls,hits,prunes,failed.Count);
    }
}


Duplicate denomination normalization is appropriate for unlimited coins; deduplicating equal-valued invoice positions would lose inventory. The state and input policy follow the business constraints.

```{index} Compare DP with an Independent Small Reference
```

## Compare DP with an Independent Small Reference




```{index} Compare DP with an Independent Small Reference
```

Breadth-first exploration of amounts starts at zero and adds one allowed coin at each level. The first discovery of target has the minimum number of coins because every transition costs one. This independent reference can check a small DP instance without reimplementing its amount-order recurrence.

In [ ]:
using System;
using System.Linq;
using System.Collections.Generic;

int[] sizes={1,3,4};int target=9;
var distance=new Dictionary<int,int>{{0,0}};var queue=new Queue<int>();queue.Enqueue(0);
while(queue.Count>0)
{
 int amount=queue.Dequeue();
 foreach(int size in sizes)
 {
  int next=amount+size;
  if(next>target||distance.ContainsKey(next))continue;
  distance[next]=distance[amount]+1;queue.Enqueue(next);
 }
}
var plan=CoinPlanning.MinimumCoins(sizes,target);
Console.WriteLine($"DP={plan.Count}, level-search={distance[target]}, witness={string.Join("+",plan.Coins)}");
if(plan.Count!=distance[target])throw new Exception("Reference disagreement.");
public static class CoinPlanning
{
    public record CoinPlan(int? Count,int[] Coins,int?[] Best);
    // Unlimited positive denominations. A target cap bounds table allocation.
    public static CoinPlan MinimumCoins(int[] denominations,int target)
    {
        ArgumentNullException.ThrowIfNull(denominations);
        if(target<0||target>1_000_000)throw new ArgumentOutOfRangeException(nameof(target));
        if(denominations.Any(c=>c<=0))throw new ArgumentException("Coins must be positive.");
        int[] coins=denominations.Distinct().OrderBy(c=>c).ToArray();
        var best=new int?[target+1];var last=new int[target+1];best[0]=0;
        for(int amount=1;amount<=target;amount++)
        {
            foreach(int coin in coins)
            {
                if(coin>amount)break;
                if(best[amount-coin] is not int count)continue;
                int candidate=count+1;
                if(best[amount] is null||candidate<best[amount])
                {best[amount]=candidate;last[amount]=coin;}
            }
        }
        var chosen=new List<int>();
        if(best[target] is not null)
            for(int remaining=target;remaining>0;remaining-=last[remaining])chosen.Add(last[remaining]);
        return new CoinPlan(best[target],chosen.ToArray(),best);
    }
}


Keep reference targets small and verify unreachable cases as well. Chapter 20 develops breadth-first traversal systematically. Here the level count provides a separate way to validate the minimum-count objective.

```{rubric} Footnotes
```


[^2501_memoization_tabulation-1]: A correct state contains all information relevant to future choices. Reusing results for different states can silently change the problem.
[^2501_memoization_tabulation-2]: Complexity claims here count fixed-width arithmetic as constant time; arbitrary-precision counting requires accounting for growing integer sizes.

[^development-reading]: Further reading: [OpenDSA, Dynamic Programming](https://opendsa.org/OpenDSA/Books/CS4104/html/DynamicProgramming.html). The worked business scenarios and guided reasoning here are original teaching examples.
